# Location selection

In [362]:
from desdeo.problem import Constant, Variable, Problem, Objective, VariableTypeEnum, Constraint, TensorConstant, TensorVariable, ConstraintTypeEnum
import numpy as np

# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Model inputs

In [363]:
# Mininum expected attendance to be worth visiting 
min_att = 15

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.00

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 6.0
# How long the event is (hours)
hours_per_event = 4
driver_salary_per_hour = 19
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour

# Food desert threshold (miles) (TODO this should be in minutes)
raw_food_desert_threshold = 15



## Load and process constants


In [364]:
from slugify import slugify
import pandas as pd

def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)

home = "Ada"
# Read the adjacency matrix (distance in miles)
adjDist = pd.read_csv("adjacencyMatrixDist.csv", index_col=0)
dist2home = adjDist.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"
display(dist2home)

# Read the adjacency matrix (travel time in minutes)
adjTTime = pd.read_csv("adjacencyMatrixTravelTime.csv", index_col=0)
display(adjTTime)

# Read the cities 
cities = pd.read_csv("cities.csv")
events = pd.read_csv("events.csv")



,dist2home
city,
Ada,0.0
Lima,16.2
Kenton,15.5
Delphos,31.8
Bluffton,12.0
Spencerville,30.8
Elida,23.5
Forest,18.4
Alger,5.4


,Ada,Alger,Bluffton,Cairo,Columbus Grove,Continental,Cridersville,Delphos,Dunkirk,Elida,...,Mount Victory,New Bremen,New Knoxville,Ottawa,Ottoville,Pandora,Saint Marys,Spencerville,Wapakoneta,Waynesfield
Ada,0,9,16,20,25,48,28,34,16,30,...,34,52,46,34,37,26,45,43,36,25
Alger,9,0,22,26,31,54,29,40,20,35,...,33,53,46,40,43,32,46,45,36,19
Bluffton,16,22,0,17,15,39,25,31,23,27,...,48,49,42,23,34,13,42,40,32,32
Cairo,21,26,17,0,15,39,20,18,27,13,...,48,44,38,17,34,13,37,26,28,28
Columbus Grove,26,31,15,9,0,26,26,24,32,20,...,57,50,43,9,22,8,43,34,33,33
Continental,48,53,40,30,25,0,49,30,54,33,...,80,68,64,19,21,32,58,42,56,56
Cridersville,28,29,25,21,26,49,0,29,39,17,...,49,30,23,35,34,30,23,18,13,17
Delphos,34,39,31,18,24,30,29,0,40,12,...,66,40,35,32,11,33,30,14,33,42
Dunkirk,16,20,23,27,32,54,39,41,0,36,...,28,63,56,41,44,33,56,50,46,36
Elida,30,35,26,13,21,33,17,12,36,0,...,61,41,33,28,17,29,32,16,24,33


In [365]:

# Create event table
events = pd.merge(cities, events, on="city")
events.loc[:,"expectedAttendance"] = (events.loc[:,"pop"] * events.loc[:,"attendanceRate"]).astype(int)

events.loc[:, "event_id"] = events.apply(lambda row: slugify(f'{row["city"]} {row["site"]} {no_nan(row["event"])}'), axis=1)

# Add the distance to home for each event 
events = pd.merge(events, dist2home, on="city")

events

,city,lat,long,pop,site,event,attendanceRate,expectedAttendance,event_id,dist2home
0,Ada,40.768056,-83.825278,5334,Public Library,NaN,0.002,10,ada-public-library,0.0
1,Lima,40.746389,-84.123333,35579,Mercy Health Thrift,NaN,0.002,71,lima-mercy-health-thrift,16.2
2,Lima,40.746389,-84.123333,35579,Habitat For Humanity,NaN,0.002,71,lima-habitat-for-humanity,16.2
3,Lima,40.746389,-84.123333,35579,Our Daily Bread,NaN,0.002,71,lima-our-daily-bread,16.2
4,Lima,40.746389,-84.123333,35579,St. Mark’s Methodist,Community Meal,0.002,71,lima-st-marks-methodist-community-meal,16.2
5,Lima,40.746389,-84.123333,35579,Christian Corner Community Center,NaN,0.002,71,lima-christian-corner-community-center,16.2
6,Kenton,40.646667,-83.622500,7947,Seton Hall,NaN,0.002,15,kenton-seton-hall,15.5
7,Kenton,40.646667,-83.622500,7947,Hardincrest,NaN,0.002,15,kenton-hardincrest,15.5
8,Kenton,40.646667,-83.622500,7947,YMCA,NaN,0.002,15,kenton-ymca,15.5
9,Delphos,40.861111,-84.350000,7117,Public Library,NaN,0.002,14,delphos-public-library,31.8


### Close cities

In [366]:
close_cities = adjTTime < raw_food_desert_threshold


# Constants

In [367]:
event_count = events.shape[0]

# Expected attenance 
raw_attendance = events.loc[:,"expectedAttendance"] 
raw_over_attendance = raw_attendance < min_att

# Expected attendance 
exp_att = []
# Over staffed events (ose)
ose = []
# Distance to home
d2h = []

# Close cities
cc = []

for e in range(event_count): 

    # TODO change the simple numbers to event ids
    city = events.loc[e,"city"]
    event_id = events.loc[e,"event_id"]

    exp_att.append(Constant(name=f"Expected attendance: {event_id}", 
                        symbol=f"exp_att_{e}", 
                        type="int", 
                        value=raw_attendance[e]
                        ))

    ose.append(Constant(name=f"Over attendance: {event_id}", 
                        symbol=f"ose_{e}", 
                        type="binary",
                        value=raw_over_attendance[e]))

    d2h.append(Constant(name=f"Distance to home: {event_id}", 
                              symbol=f"d2h_{e}", 
                              type="real", 
                              value=dist2home.loc[city]
                              ))

    cc.append(TensorConstant(name=f"Close cities: {event_id}", 
                       symbol=f"cc_{e}",
                       type="integer",
                       shape=[close_cities.shape[0]],
                       values=(close_cities.loc[:,city].values).astype(int).tolist()
                    ))    


mpg = Constant(name="Miles per gallon (miles/gallon)", 
               symbol="mpg", 
               type="real",
               value=raw_mpg)

dpg = Constant(name="Dollars per gallon ($/gallon)", 
               symbol="dpg", 
               type="real",
               value=raw_dollars_per_gallon)

driver_cost_per_trip = Constant(name="Driver cost per trip ($)", 
                                symbol="dcpt", 
                                type="real", 
                                value=raw_driver_cost_per_trip)


food_desert_threshold = Constant(name="Food desert threshold (miles)", 
                                 symbol="fdt", 
                                 type="real",
                                 value=raw_food_desert_threshold
                                 )
cc

[TensorConstant(name='Close cities: ada-public-library', symbol='cc_0', shape=[28], values=['List', 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 TensorConstant(name='Close cities: lima-mercy-health-thrift', symbol='cc_1', shape=[28], values=['List', 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 TensorConstant(name='Close cities: lima-habitat-for-humanity', symbol='cc_2', shape=[28], values=['List', 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 TensorConstant(name='Close cities: lima-our-daily-bread', symbol='cc_3', shape=[28], values=['List', 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 TensorConstant(name='Close cities: lima-st-marks-methodist-community-meal', symbol='cc_4', shape=[28], values=['List', 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 TensorConstant(name='Close cities:

## Constraints
Making a helper constraint to model an boolean OR operation

| $x_{Ada}$ | $x_{Lima_1}$ | $x_{Lima_2}$ | $z$ |
| --------- | ------------ | ------------ | --- |
| 1         | 1            | 1            | 1   |
| 1         | 1            | 0            | 1   |
| 1         | 0            | 1            | 1   |
| 1         | 0            | 0            | 1   |
| 0         | 1            | 1            | 1   |
| 0         | 1            | 0            | 1   |
| 0         | 0            | 1            | 1   |
| 0         | 0            | 0            | 0   |

Constraints to add: 

$z \ge x_{Ada}$, or  $0 \ge x_{Ada} - z$

$z \ge x_{Lima_1}$, or $0 \ge x_{Lima_1} - z$

$z \ge x_{Lima_2}$, or $0 \ge x_{Lima_2} - z$

$z \le \sum{x_i}$, or $z - (\sum{x_i}) \le 0 $




In [368]:
desert_or_constr_exprs = []

for e in range(events.shape[0]): 
    desert_or_constr_exprs.append(
        f"evt_visit_{e} * cc_{e} - z"
    )

final_const_exprs = " + ".join([f"event_visit_{e} * cc_{e}" for e in range(events.shape[0]) ]) 
final_const_exprs = f"z-({final_const_exprs})"



desert_or_constr_exprs.append(final_const_exprs)
desert_or_constr_exprs

desert_or_constr_exprs

['evt_visit_0 * cc_0 - z',
 'evt_visit_1 * cc_1 - z',
 'evt_visit_2 * cc_2 - z',
 'evt_visit_3 * cc_3 - z',
 'evt_visit_4 * cc_4 - z',
 'evt_visit_5 * cc_5 - z',
 'evt_visit_6 * cc_6 - z',
 'evt_visit_7 * cc_7 - z',
 'evt_visit_8 * cc_8 - z',
 'evt_visit_9 * cc_9 - z',
 'evt_visit_10 * cc_10 - z',
 'evt_visit_11 * cc_11 - z',
 'evt_visit_12 * cc_12 - z',
 'evt_visit_13 * cc_13 - z',
 'evt_visit_14 * cc_14 - z',
 'evt_visit_15 * cc_15 - z',
 'evt_visit_16 * cc_16 - z',
 'evt_visit_17 * cc_17 - z',
 'z-(event_visit_0 * cc_0 + event_visit_1 * cc_1 + event_visit_2 * cc_2 + event_visit_3 * cc_3 + event_visit_4 * cc_4 + event_visit_5 * cc_5 + event_visit_6 * cc_6 + event_visit_7 * cc_7 + event_visit_8 * cc_8 + event_visit_9 * cc_9 + event_visit_10 * cc_10 + event_visit_11 * cc_11 + event_visit_12 * cc_12 + event_visit_13 * cc_13 + event_visit_14 * cc_14 + event_visit_15 * cc_15 + event_visit_16 * cc_16 + event_visit_17 * cc_17)']

In [369]:
constraints = []
for (e, expr) in enumerate(desert_or_constr_exprs): 
  constraints.append(
    Constraint(
      name=f"Desert constraint #{e}",
      symbol=f"ds_{e}",
      func=expr,
      cons_type=ConstraintTypeEnum.LTE,
      is_convex=False,
      is_linear=True,
      is_twice_differentiable=True)
)

## Variables
Generate one variable per event

In [370]:
# Main variables (visit event, or don't visit event)

evt_visited = []
for e in range(event_count):
  evt_visited.append(Variable(
    name=events.loc[e, "event_id"],
    symbol=f"evt_visit_{e}",
    variable_type=VariableTypeEnum.integer,
    lowerbound=0,
    upperbound=1,
    initial_value=0))
  
# Auxiliary variables 
aux_desert_var = TensorVariable(
  name="Desert auxiliary variable",
  symbol="z",
  variable_type=VariableTypeEnum.integer,
  shape=[close_cities.shape[0]],
  lowerbounds=0,
  upperbounds=events.shape[0]
)
aux_desert_var

TensorVariable(name='Desert auxiliary variable', symbol='z', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[28], lowerbounds=0, upperbounds=18, initial_values=None)

## Objectives

### Build the objective expression strings

In [371]:
# total patients served
obj_func_total_patients = " + ".join([f"evt_visit_{e} * exp_att_{e}" for e in range(event_count)])
display(obj_func_total_patients)


'evt_visit_0 * exp_att_0 + evt_visit_1 * exp_att_1 + evt_visit_2 * exp_att_2 + evt_visit_3 * exp_att_3 + evt_visit_4 * exp_att_4 + evt_visit_5 * exp_att_5 + evt_visit_6 * exp_att_6 + evt_visit_7 * exp_att_7 + evt_visit_8 * exp_att_8 + evt_visit_9 * exp_att_9 + evt_visit_10 * exp_att_10 + evt_visit_11 * exp_att_11 + evt_visit_12 * exp_att_12 + evt_visit_13 * exp_att_13 + evt_visit_14 * exp_att_14 + evt_visit_15 * exp_att_15 + evt_visit_16 * exp_att_16 + evt_visit_17 * exp_att_17'

In [372]:

# ose = over staffed events
obj_func_ose = " + ".join([f"evt_visit_{e} * ose_{e}" for e in range(event_count)])
display(obj_func_ose)


'evt_visit_0 * ose_0 + evt_visit_1 * ose_1 + evt_visit_2 * ose_2 + evt_visit_3 * ose_3 + evt_visit_4 * ose_4 + evt_visit_5 * ose_5 + evt_visit_6 * ose_6 + evt_visit_7 * ose_7 + evt_visit_8 * ose_8 + evt_visit_9 * ose_9 + evt_visit_10 * ose_10 + evt_visit_11 * ose_11 + evt_visit_12 * ose_12 + evt_visit_13 * ose_13 + evt_visit_14 * ose_14 + evt_visit_15 * ose_15 + evt_visit_16 * ose_16 + evt_visit_17 * ose_17'

In [373]:
# Total costs
total_distance = " + ".join([f"evt_visit_{e} * d2h_{e}" for e in range(event_count)])
print("Total distance:")
display(total_distance)

print("Total gas cost")
total_gas_costs = f"(({total_distance})/mpg)*dpg"
display(total_gas_costs)

# TODO I can represent the drive time of the driver here once I have the drive distance matrix
print("Total driver cost")
total_driver_cost = " + ".join([f"evt_visit_{e}" for e in range(event_count)])
total_driver_cost = f"({total_driver_cost})*dcpt"

display(total_driver_cost)

print("Total costs")
obj_total_cost = f"{total_gas_costs} + {total_driver_cost}"
display(obj_total_cost)


Total distance:


'evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17'

Total gas cost


'((evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17)/mpg)*dpg'

Total driver cost


'(evt_visit_0 + evt_visit_1 + evt_visit_2 + evt_visit_3 + evt_visit_4 + evt_visit_5 + evt_visit_6 + evt_visit_7 + evt_visit_8 + evt_visit_9 + evt_visit_10 + evt_visit_11 + evt_visit_12 + evt_visit_13 + evt_visit_14 + evt_visit_15 + evt_visit_16 + evt_visit_17)*dcpt'

Total costs


'((evt_visit_0 * d2h_0 + evt_visit_1 * d2h_1 + evt_visit_2 * d2h_2 + evt_visit_3 * d2h_3 + evt_visit_4 * d2h_4 + evt_visit_5 * d2h_5 + evt_visit_6 * d2h_6 + evt_visit_7 * d2h_7 + evt_visit_8 * d2h_8 + evt_visit_9 * d2h_9 + evt_visit_10 * d2h_10 + evt_visit_11 * d2h_11 + evt_visit_12 * d2h_12 + evt_visit_13 * d2h_13 + evt_visit_14 * d2h_14 + evt_visit_15 * d2h_15 + evt_visit_16 * d2h_16 + evt_visit_17 * d2h_17)/mpg)*dpg + (evt_visit_0 + evt_visit_1 + evt_visit_2 + evt_visit_3 + evt_visit_4 + evt_visit_5 + evt_visit_6 + evt_visit_7 + evt_visit_8 + evt_visit_9 + evt_visit_10 + evt_visit_11 + evt_visit_12 + evt_visit_13 + evt_visit_14 + evt_visit_15 + evt_visit_16 + evt_visit_17)*dcpt'

### Create the objective objects

In [374]:

# Total patients visited 
total_patients = Objective(
    name = "Maximize total patients visited",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = obj_func_total_patients
)

# Overstaffed events
over_staffed_events = Objective(
    name = "Minimize the number of overstaffed events",
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = obj_func_ose
)

# Total costs
costs = Objective(
    name = "Minimize the total costs",
    symbol = "f_3",
    maximize = False,
    is_twice_differentiable=True,
    func = obj_total_cost
)

# Communities served
comm_served = Objective(
    name = "Maximize the number of communities served",
    symbol = "f_4",
    maximize = True,
    is_twice_differentiable=True,
    func = "Sum(z)"
)

## Problem 


In [375]:
 

prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        type="linear",
        constants=exp_att +  ose + d2h + cc + [mpg, dpg, driver_cost_per_trip, food_desert_threshold],
        variables=evt_visited + [aux_desert_var],
        objectives=[total_patients, over_staffed_events, costs, comm_served],
        constraints=constraints
    )

## Ideal/nadir


In [376]:
# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(events.loc[:,"expectedAttendance"]))

# f_2 ideal is having no over staffed events
# f_2 naird is having visiting all locations with over staffed events
all_ose = int(np.sum(raw_over_attendance))

# f_3 ideal is no costs
# f_3 nadir is going to every site

# How many miles are driven/gas costs
max_dist = events.loc[:,"dist2home"].sum()
total_gas_cost = (max_dist / raw_mpg) * raw_dollars_per_gallon
driver_cost = events.shape[0]*raw_driver_cost_per_trip
max_costs = float(driver_cost + total_gas_cost)

# f_4 ideal is all cities, nadir no cities


prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0,
        "f_3": 0,
        "f_4": len(adjTTime.index)
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose,
        "f_3": max_costs,
        "f_4": 0
        }
    )

print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")

Ideal values: {'f_1': 441, 'f_2': 0, 'f_3': 0, 'f_4': 28}
Nadir values: {'f_1': 0, 'f_2': 10, 'f_3': 1506.95, 'f_4': 0}


## Solve (with NSGA-III)!

In [ ]:
from desdeo.emo.methods.EAs import nsga3_mixed_integer


solver, publisher = nsga3_mixed_integer(problem=prob)

result = solver()



### Clean up NSGA-III results 

In [ ]:
pf_obj = (result.outputs.to_pandas()).astype(int).loc[:,["f_1", "f_2", "f_3"]]
pf_dec = (result.solutions.to_pandas()).astype(bool)

results = pd.DataFrame(pf_obj.values, 
                       columns=["Total Patients Served", "Number of over staffed events", "Total Costs"])

# Build string lists for the events visited
pf_events_visited = []
for (e, ev) in pf_dec.iterrows(): 
    events_visited = ev.values
    pf_events_visited.append("\n".join(events.loc[events_visited, "event_id"].values))


results.loc[:,"events_visited"] = pf_events_visited

results = results.drop_duplicates()

results

## Plot plot plot

In [ ]:
%matplotlib qt

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

x = results.loc[:,"Total Patients Served"].values
y = results.loc[:,"Number of over staffed events"].values
z = results.loc[:,"Total Costs"].values

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(x, y, z,
           linewidths=1, alpha=.7,
           edgecolor='k',
           s = 200)

ax.set_xlabel("Total patients served")
ax.set_ylabel("Number of over staffed events")
ax.set_zlabel("Total costs")

#cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
#cbar.set_label('Total Travel Costs ($)')  # Label for the color bar

plt.show()

## Solve! (with reference point method)

In [378]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions

reference_point = {"f_1": 200, "f_2": 2, "f_3": 1000, "f_4": 20}
display(prob)
results = rpm_solve_solutions(prob, reference_point=reference_point)


Problem(name='Simple site selection', description='Simple implementation of the site selection problem', constants=[Constant(name='Expected attendance: ada-public-library', symbol='exp_att_0', value=10.0), Constant(name='Expected attendance: lima-mercy-health-thrift', symbol='exp_att_1', value=71.0), Constant(name='Expected attendance: lima-habitat-for-humanity', symbol='exp_att_2', value=71.0), Constant(name='Expected attendance: lima-our-daily-bread', symbol='exp_att_3', value=71.0), Constant(name='Expected attendance: lima-st-marks-methodist-community-meal', symbol='exp_att_4', value=71.0), Constant(name='Expected attendance: lima-christian-corner-community-center', symbol='exp_att_5', value=71.0), Constant(name='Expected attendance: kenton-seton-hall', symbol='exp_att_6', value=15.0), Constant(name='Expected attendance: kenton-hardincrest', symbol='exp_att_7', value=15.0), Constant(name='Expected attendance: kenton-ymca', symbol='exp_att_8', value=15.0), Constant(name='Expected att

AttributeError: 'FiniteScalarRangeSet' object has no attribute 'set_tuple'

In [ ]:
for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print("---")

results


Solution 1:
Objective function values 		 {'f_1': 355.0, 'f_2': 0.0, 'f_3': 420.5}
Decision variable values 		 {'evt_visit_0': 0.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 0.0, 'evt_visit_7': 0.0, 'evt_visit_8': 0.0, 'evt_visit_9': 0.0, 'evt_visit_10': 0.0, 'evt_visit_11': 0.0, 'evt_visit_12': 0.0, 'evt_visit_13': 0.0, 'evt_visit_14': 0.0, 'evt_visit_15': 0.0, 'evt_visit_16': 0.0, 'evt_visit_17': 0.0, '_alpha': -0.19999998999000201}
Constraint values 			 {'f_1_con': -0.1514739321154993, 'f_2_con': 9.989999993909038e-09, 'f_3_con': -0.18455158744484873}
---
Solution 2:
Objective function values 		 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 1506.95}
Decision variable values 		 {'evt_visit_0': 1.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 1.0, 'evt_visit_7': 1.0, 'evt_visit_8': 1.0, 'evt_visit_9': 1.0, 'evt_visit_10': 1.0, 'evt_visit_11': 1.0, 'evt_vi

[SolverResults(optimal_variables={'evt_visit_0': 0.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 0.0, 'evt_visit_7': 0.0, 'evt_visit_8': 0.0, 'evt_visit_9': 0.0, 'evt_visit_10': 0.0, 'evt_visit_11': 0.0, 'evt_visit_12': 0.0, 'evt_visit_13': 0.0, 'evt_visit_14': 0.0, 'evt_visit_15': 0.0, 'evt_visit_16': 0.0, 'evt_visit_17': 0.0, '_alpha': -0.19999998999000201}, optimal_objectives={'f_1': 355.0, 'f_2': 0.0, 'f_3': 420.5}, constraint_values={'f_1_con': -0.1514739321154993, 'f_2_con': 9.989999993909038e-09, 'f_3_con': -0.18455158744484873}, extra_func_values=None, scalarization_values={'_asf': -0.20000051593821658}, success=True, message="Pyomo solver status is: 'ok', with termination condition: 'optimal'."),
 SolverResults(optimal_variables={'evt_visit_0': 1.0, 'evt_visit_1': 1.0, 'evt_visit_2': 1.0, 'evt_visit_3': 1.0, 'evt_visit_4': 1.0, 'evt_visit_5': 1.0, 'evt_visit_6': 1.0, 'evt_visit_7': 1.0, 'evt_visit_8': 1.0,